# Extended TWFE in ModernDiD and R

This notebook is generated from the [Extended TWFE in ModernDiD and R](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_etwfe.html) page of the ModernDiD documentation.

This comparison checks whether [`etwfe`](https://moderndid.readthedocs.io/en/stable/api/generated/etwfe/moderndid.etwfe.html) and
[`emfx`](https://moderndid.readthedocs.io/en/stable/api/generated/etwfe/moderndid.emfx.html) give the same answers as Grant McDermott's R package
`etwfe` on the minimum wage data. Both languages fit the regression from the
[extended TWFE example](https://moderndid.readthedocs.io/en/stable/user_guide/example_etwfe.html#example-etwfe), once with not-yet-treated
controls and once with never-treated controls. For each design we compare
every cohort-year coefficient and its standard error before turning to the
simple, group, calendar, and event aggregations. Through `rpy2`, the R calls
run in the same Python session as ModernDiD.

Under both designs, the coefficients, their clustered standard errors, and
every aggregated estimate agree to within 1e-14. Because R's `emfx`
differentiates each average numerically, the standard errors of the
aggregations differ by up to 2e-8. Among the calls on this page, the two
packages disagree only when a treated cohort is the reference. ModernDiD
rejects such a cohort under never-treated controls and names the alternatives
in its error message. R accepts it under either design and then counts the
never-treated counties, coded 0 in this data, as treated. In the never-treated
design of Wooldridge (2025), those counties are the controls for every cohort.
The [section on reference cohorts](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_etwfe.html#a-treated-reference-cohort) traces R's
behavior to the lines of its source.

> **What this comparison checks**
>
> Every fit on this page is a linear regression whose only covariate is log
> population. The R reference tests in `tests/validation/test_r_etwfe.py` also
> cover the Poisson, logit, and probit families, `xvar`, and the `"feo"` and
> `"none"` fixed-effect settings.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    # Colab ships rpy2 3.5 as a single package. Uninstalling it first keeps pip from deleting
    # the files it shares with the split rpy2 3.6 packages.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "rpy2"], check=True)
    # rpy2 supports Colab's R 4.6 from release 3.6.7 on.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "moderndid[all]==0.4.1", "rpy2>=3.6.7"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-cran-etwfe", "r-cran-fixest"], check=True)
else:
    print("Outside Colab, set up R as https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running describes.")

## One county panel for both packages

[`load_mpdta`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_mpdta.html) loads the county panel that the extended TWFE
example studies. Its 500 US counties each appear once a year from 2003
through 2007. Each county-year row holds log teen employment (`lemp`), log
population measured in 2000 (`lpop`), and the year its state first raised
the minimum wage (`first.treat`). The 309 counties coded 0 in `first.treat`
lie in states that did not raise their minimum wage in these years and form
the never-treated controls. The other 191 adopted in 2004 (20 counties),
2006 (40), or 2007 (131).

Because we pass R the data unchanged, both fits start from the same 2,500
county-years. The section on
[running the comparisons](https://moderndid.readthedocs.io/en/stable/r_comparisons/index.html#r-comparisons-running) explains how to
install the R packages and rerun this notebook.

In [ ]:
import rpy2.robjects as ro

import moderndid as did

data = did.load_mpdta()

In [ ]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# The page compares standard errors that differ only in the eighth decimal.
pd.set_option("display.precision", 8)

with ro.default_converter.context():
    importr("etwfe")
    # A wider console keeps each row of R's coefficient tables on one line.
    ro.r("options(width = 120)")

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["mpdta"] = ro.conversion.py2rpy(data.to_pandas())

## The not-yet-treated regression

We begin with the regression that the extended TWFE example fits. Under
`cgroup="notyet"`, the controls are the never-treated counties plus each
cohort's county-years before adoption. Every treated cohort-year, seven in
all, enters with its own indicator and that indicator's interaction with
`lpop` centered within the cohort. The regression also allows a separate
slope on `lpop` in each year and absorbs county and year fixed effects. Both
calls cluster the standard errors by county, as the example does.

In [ ]:
# The extended TWFE example's regression under not-yet-treated controls.
spec = dict(
    yname="lemp",
    tname="year",
    gname="first.treat",
    idname="countyreal",
    cgroup="notyet",
    xformla="~lpop",
    vcov={"CRV1": "countyreal"},
)
py_notyet = did.etwfe(data, **spec)
print(py_notyet)

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_notyet <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "notyet",
            vcov = ~countyreal
        )
        print(summary(r_notyet))
    """)

Besides the seven cells, R's summary lists the four year-specific slopes on
`lpop` and each cell's interaction with centered `lpop`. It also reports 14
terms that the regression dropped as collinear. Ten of them belong to
cohort-years before adoption, where the treatment indicator is always off.
The other four are slopes on `lpop` that the county fixed effects absorb,
since `lpop` never changes within a county. ModernDiD builds no columns for
the first ten and drops the same four slopes. It keeps all 18 estimated
terms in `coef_names` and `model_coefficients` but prints only the seven
cells.

The last lines of R's summary report the adjusted R-squared of the
regression. The table below sets R's two R-squared statistics beside the
`r_squared` and `adj_r_squared` fields of ModernDiD's result.

In [ ]:
# Since both statistics come from the residuals, they also check the fixed effects.
with ro.default_converter.context():
    r_fit_statistics = np.asarray(ro.r('fixest::r2(r_notyet, c("r2", "ar2"))'))
fit_statistics = pd.DataFrame(
    {"Python": [py_notyet.r_squared, py_notyet.adj_r_squared], "R": r_fit_statistics},
    index=["R-squared", "adjusted R-squared"],
)
fit_statistics["difference"] = fit_statistics["Python"] - fit_statistics["R"]
fit_statistics

With R-squared and adjusted R-squared equal to the last digit, both fits
explain the same share of the variation in teen employment. The next table
finds each cell among R's coefficient names and joins the two fits by
cohort and year rather than by position. Its difference columns subtract
each R value from the Python value before any rounding.

In [ ]:
def compare_cells(python_fit, r_model):
    """Set each cohort-year coefficient beside its R counterpart."""
    python_cells = pd.DataFrame({
        "cohort": [int(g) for g, _ in python_fit.gt_pairs],
        "year": [int(t) for _, t in python_fit.gt_pairs],
        "Python estimate": python_fit.coefficients,
        "Python SE": python_fit.std_errors,
    })
    with ro.default_converter.context():
        names = list(ro.r(f"names(coef({r_model}))"))
        r_estimates = np.asarray(ro.r(f"coef({r_model})"))
        r_se = np.asarray(ro.r(f"fixest::se({r_model})"))

    # R names each cell ".Dtreat:first.treat::<cohort>:year::<year>".
    r_cells = pd.Series(names).str.extract(r"^\.Dtreat:first\.treat::(?P<cohort>\d+):year::(?P<year>\d+)$")
    r_cells["R estimate"] = r_estimates
    r_cells["R SE"] = r_se
    r_cells = r_cells.dropna(subset=["cohort"]).astype({"cohort": int, "year": int})

    cells = python_cells.merge(r_cells, on=["cohort", "year"], validate="one_to_one")
    assert len(cells) == len(python_cells) == len(r_cells), "The two fits returned different cohort-year cells."
    cells["estimate difference"] = cells["Python estimate"] - cells["R estimate"]
    cells["SE difference"] = cells["Python SE"] - cells["R SE"]
    return cells


notyet_cells = compare_cells(py_notyet, "r_notyet")
notyet_cells

Every coefficient agrees to within 3.1e-15 and every standard error to
within 7.1e-16. The two packages therefore fit the same regression and
apply the same small-sample correction to its county-clustered variance
matrix.

## Averaging the cells with emfx

[`emfx`](https://moderndid.readthedocs.io/en/stable/api/generated/etwfe/moderndid.emfx.html) and R's `emfx` both turn the cells into four kinds
of averages. `type="simple"` averages over every treated county-year,
`"group"` within each cohort, `"calendar"` within each year, and `"event"`
within each number of years since the increase. In every average that both
packages report, they weight each cell by its number of county-years.
ModernDiD writes each average as a fixed combination of the coefficients.
R's `emfx` instead hands the treated rows to the `slopes` function of
`marginaleffects`. That function predicts each row's outcome with the
treatment indicator switched on and off and averages the differences over
the rows behind each estimate.

Below, each package prints its event study in its own format before one
table sets all four aggregations side by side.

In [ ]:
py_notyet_event = did.emfx(py_notyet, type="event")
print(py_notyet_event)

In [ ]:
with ro.default_converter.context():
    ro.r('print(etwfe::emfx(r_notyet, type = "event"))')

ModernDiD's summary rows for the group, calendar, and event types, such as
the event study's −0.0845 above, have no counterpart in R's `emfx`. The
table below therefore compares the simple average and the rows by cohort,
year, and event time. Its `by` column names the cohort, year, or event time
that each row averages over.

In [ ]:
def compare_aggregations(python_fit, r_model, post_only=True, r_arguments=""):
    """Set each emfx row beside its R counterpart for all four aggregation types."""
    tables = []
    for kind, r_by in [("simple", ".Dtreat"), ("group", "first.treat"), ("calendar", "year"), ("event", "event")]:
        python_result = did.emfx(python_fit, type=kind, post_only=post_only)
        with (ro.default_converter + pandas2ri.converter).context():
            r_rows = ro.r(f'as.data.frame(etwfe::emfx({r_model}, type = "{kind}"{r_arguments}))')
        if kind == "simple":
            # The simple average has no cohort, year, or event time to join on.
            python_rows = pd.DataFrame(
                {"by": [pd.NA], "att": [python_result.overall_att], "se": [python_result.overall_se]}
            )
            r_rows[r_by] = pd.NA
        else:
            python_rows = did.to_df(python_result).to_pandas().rename(columns={"event_time": "by"})
        # Both event studies fix the base year at zero without a standard error. Like to_df, leave that row out.
        r_rows = r_rows.dropna(subset=["std.error"]).rename(columns={r_by: "by"})
        rows = python_rows.astype({"by": "Int64"}).merge(r_rows.astype({"by": "Int64"}), on="by", validate="one_to_one")
        assert len(rows) == len(python_rows) == len(r_rows), f"The {kind} aggregations returned different rows."
        tables.append(
            pd.DataFrame({
                "aggregation": kind,
                "by": rows["by"],
                "Python estimate": rows["att"],
                "Python SE": rows["se"],
                "R estimate": rows["estimate"],
                "R SE": rows["std.error"],
            })
        )

    table = pd.concat(tables, ignore_index=True)
    table["estimate difference"] = table["Python estimate"] - table["R estimate"]
    table["SE difference"] = table["Python SE"] - table["R SE"]
    return table


notyet_aggregations = compare_aggregations(py_notyet, "r_notyet")
notyet_aggregations

Like the cells, every aggregated estimate agrees with R's to within 3.0e-15.
The standard errors differ in the eighth to tenth decimal place instead,
most of all in the calendar row for 2004 at 1.8e-8.


### Standard errors in the eighth decimal

ModernDiD takes each standard error from the exact gradient of an average
with respect to the coefficients. In R, `marginaleffects` nudges one
coefficient at a time, recomputes the averages, and approximates the
gradient by forward differences. The calendar row for 2004 averages the
single cell (2004, 2004). Its standard error must therefore equal that
cell's regression standard error, 0.02169330 in both packages. ModernDiD's
calendar row shows the same 0.02169330, against 0.02169328 in R.

Passing `numderiv = "richardson"` through `emfx` to `slopes` replaces the
forward differences with Richardson extrapolation. For each aggregation
type, the following table compares the largest standard-error gap under R's
default derivative with the largest under Richardson extrapolation.

In [ ]:
notyet_richardson = compare_aggregations(py_notyet, "r_notyet", r_arguments=', numderiv = "richardson"')
gaps = pd.DataFrame({
    "aggregation": notyet_aggregations["aggregation"],
    "largest SE difference, forward differences": notyet_aggregations["SE difference"].abs(),
    "largest SE difference, Richardson": notyet_richardson["SE difference"].abs(),
})
gaps.groupby("aggregation", sort=False).max()

Since Richardson extrapolation pushes every gap below 2e-11, the gaps above
come from R's numerical derivative rather than from the variance matrix or
the cell weights.

> **Tighten R's aggregated standard errors**
>
> When you check ModernDiD against R on your own data, pass
> `numderiv = "richardson"` to `emfx`. That leaves the estimates unchanged
> and, on the minimum wage data, brings the standard errors into agreement to
> about ten decimal places.

## The never-treated regression

With `cgroup="never"`, the never-treated counties are the only controls for
every cohort. Each cohort's effects are measured against its year before the
increase. Because each cohort gets a cell in every year except that one, the
2006 and 2007 cohorts add five placebo cells before their increases, for 12
cells in all. Both calls below change only the control group and keep every
other argument of the not-yet-treated fits.

In [ ]:
py_never = did.etwfe(data, **(spec | {"cgroup": "never"}))
print(py_never)

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_never <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "never",
            vcov = ~countyreal
        )
        print(summary(r_never))
    """)

This time R's summary reports 10 terms that the regression dropped as
collinear. Besides the four slopes on `lpop`, they are each cohort's
base-year cell and that cell's interaction with centered `lpop`. With the
treatment indicator off in every base year, those columns hold only zeros.
The 12 estimated cells line up by cohort and year as before.

In [ ]:
never_cells = compare_cells(py_never, "r_never")
never_cells

The placebo cells agree as closely as the cells after adoption. Every
estimate lies within 6.3e-15 of R's and every standard error within
2.0e-15.

The never-treated event study differs between the two packages in one
default. R's documentation says that `emfx` evaluates `post_only` only
under not-yet-treated controls. Under never-treated controls, its event
study therefore always reports the placebo event times −4 to −2 and the
base year −1 (lines 190 to 196 of `R/emfx.R` in `etwfe` 0.6.2). The table
leaves event time −1 out because both event studies fix the base year at
zero without a standard error.

> **Match R's rows with post_only=False**
>
> ModernDiD's `emfx` keeps only the event times from zero on unless you pass
> `post_only=False`, as the calls below do.

In [ ]:
py_never_event = did.emfx(py_never, type="event", post_only=False)
print(py_never_event)

In [ ]:
with ro.default_converter.context():
    ro.r('print(etwfe::emfx(r_never, type = "event"))')

In [ ]:
never_aggregations = compare_aggregations(py_never, "r_never", post_only=False)
never_aggregations

The never-treated aggregations repeat the pattern that the not-yet-treated
ones showed. Estimates agree to within 6.4e-15 and standard errors to
within 1.8e-8, again largest in the calendar row for 2004.


## A treated reference cohort

Both designs above leave `gref` unset and take the never-treated counties
as the reference cohort. Under `cgroup="never"`, the design of
[Wooldridge (2025)](https://doi.org/10.1007/s00181-025-02807-z) compares
every cohort with the never-treated units. ModernDiD therefore accepts only
a never-treated code for `gref` there and raises an error for a treated
cohort such as 2007.

In [ ]:
try:
    did.etwfe(data, **(spec | {"cgroup": "never", "gref": 2007}))
except Exception as error:
    print(f"{type(error).__name__}: {error}")

### What R estimates under never-treated controls

In R, `etwfe` fits the same call without an error and returns estimates for
it. The output below shows R's simple average and group aggregation for that
fit.

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_treated_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "never",
            gref = 2007,
            vcov = ~countyreal
        )
        print(etwfe::emfx(r_treated_reference))
        print(etwfe::emfx(r_treated_reference, type = "group"))
    """)

The group table lists cohort 0, the never-treated counties, among the
treated cohorts. Line 355 of `R/etwfe.R` checks only that a supplied `gref`
occurs in `first.treat`. Line 385 then switches the treatment indicator on
in every year except the one before adoption. Since counties coded 0 have
no such year, the indicator stays on for them in all five years. Lines 361
and 389 make 2007 the reference level of the cohort-year interactions.
Without cells of its own, cohort 2007 stays on the control side in all five
years, even in its treated year of 2007.

In `R/emfx.R`, line 191 carries the comment "Drop never treated reference
group" but removes cohort 2007. The never-treated counties then pass the
filters on lines 194 and 195 as treated rows. R's simple average of −0.0322
mixes their 1,545 county-years with the 160 county-years after adoption in
the 2004 and 2006 cohorts.

> **R counts never-treated counties as treated**
>
> Like the simple average, R's calendar rows from 2004 on mix the never-treated
> counties' cells with the effects of the increases. R's calendar row for 2003
> and its event rows labeled 2003 to 2007 report those counties alone. Since
> none of these numbers answers a question that the never-treated design poses,
> use ModernDiD's not-yet-treated fit for a treated reference cohort, as the
> next section shows.

### The not-yet-treated route

For a treated reference cohort, ModernDiD's error message points to
`cgroup="notyet"` instead. Under not-yet-treated controls, a treated
reference cohort drops every year from its adoption on. In R, this route
depends on how the never-treated counties are coded. R reads a code before
the first year as never treated only when `gref` is unset (`R/etwfe.R`, line
334). With `gref = 2007` and the 0 coding of this data, line 375 marks the
never-treated counties as treated. The code below runs that call in R and
prints its simple and group averages.

In [ ]:
with ro.default_converter.context():
    ro.r("""
        r_notyet_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta,
            ivar = countyreal,
            cgroup = "notyet",
            gref = 2007,
            vcov = ~countyreal
        )
        print(etwfe::emfx(r_notyet_reference))
        print(etwfe::emfx(r_notyet_reference, type = "group"))
    """)

Again the group table lists cohort 0 among the treated cohorts. R estimates
its effects for 2004 through 2006 and folds them into a simple average of
−0.0108. Once the never-treated counties carry a code after the last year,
such as 9999, line 375 leaves them untreated. ModernDiD reads the 0 coding
as never treated whatever `gref` is. We therefore fit ModernDiD on the
original data and R on a copy with the never-treated counties recoded to
9999.

In [ ]:
py_reference = did.etwfe(data, **(spec | {"gref": 2007}))

In [ ]:
# R reads 9999, a code after the last year, as never treated even when gref is set.
with ro.default_converter.context():
    ro.r("""
        mpdta_9999 <- mpdta
        mpdta_9999$first.treat[mpdta_9999$first.treat == 0] <- 9999L
        r_recoded_reference <- etwfe::etwfe(
            fml = lemp ~ lpop,
            tvar = year,
            gvar = first.treat,
            data = mpdta_9999,
            ivar = countyreal,
            cgroup = "notyet",
            gref = 2007,
            vcov = ~countyreal
        )
    """)

In [ ]:
reference_cells = compare_cells(py_reference, "r_recoded_reference")
reference_cells

In [ ]:
reference_aggregations = compare_aggregations(py_reference, "r_recoded_reference")
reference_aggregations

Fit on the recoded copy, R keeps the same four cells as ModernDiD for the
2004 and 2006 cohorts and reaches the same simple average of −0.0472. The
two fits' estimates agree to within 2.1e-14 in the cells and the
aggregations alike. The aggregated standard errors again differ only through
R's forward differences, by up to 5.6e-9. On the regression without
covariates, the reference test `test_treated_reference_cohort_matches_r` in
`tests/validation/test_r_etwfe.py` checks the estimates of the same four
cells and the simple average with its standard error.

## Tolerances for a rerun

Assertions fail the notebook if a rerun moves any compared value
outside the tolerances of the
[staggered DiD comparison](https://moderndid.readthedocs.io/en/stable/r_comparisons/r_staggered.html#r-comparisons-staggered). Each estimate and
R-squared statistic may differ from R by at most
`1e-6 + 1e-5 * abs(R value)` and each standard error by at most
`1e-6 + 1e-3 * abs(R SE)`. The table below lists the largest gap in each
comparison.

In [ ]:
np.testing.assert_allclose(fit_statistics["Python"], fit_statistics["R"], rtol=1e-5, atol=1e-6)
comparisons = pd.concat(
    {
        "not-yet-treated cells": notyet_cells,
        "not-yet-treated aggregations": notyet_aggregations,
        "never-treated cells": never_cells,
        "never-treated aggregations": never_aggregations,
        "treated reference cells": reference_cells,
        "treated reference aggregations": reference_aggregations,
    },
    names=["comparison"],
)
np.testing.assert_allclose(comparisons["Python estimate"], comparisons["R estimate"], rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(comparisons["Python SE"], comparisons["R SE"], rtol=1e-3, atol=1e-6)
comparisons[["estimate difference", "SE difference"]].abs().groupby("comparison", sort=False).max()

Even the largest gap in the table is less than a thousandth of its
tolerance.

## Versions used

The table records the versions of Python, R, and the packages on each side
that produced the saved output.

In [ ]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    versions = {
        "Python": python_version(),
        "ModernDiD": version("moderndid"),
        "pyfixest": version("pyfixest"),
        "rpy2": version("rpy2"),
        "R": ro.r("R.version.string")[0],
        "etwfe": ro.r('as.character(packageVersion("etwfe"))')[0],
        "fixest": ro.r('as.character(packageVersion("fixest"))')[0],
        "marginaleffects": ro.r('as.character(packageVersion("marginaleffects"))')[0],
    }
pd.DataFrame({"component": list(versions), "version": list(versions.values())})